# Ukryte Kategorie

![Grafika do zadania](https://i.imgur.com/7o3Y4WP.png)

_Źródło: Obraz wygenerowany za pomocą ChatGPT._

## Wstęp

Kuba właśnie zaczął pracę w firmie obsługującej sklep e-commerce. Jednym z jego zadań jest opisywanie kategorii produktów. Część produktów została opisana przez jego poprzednika i firmie bardzo zależy na spójnym doborze etykiet. Niestety, nie zostawił on żadnych instrukcji. Pomóż Kubie uzupełnić brakujące etykiety.

Współczesne systemy rekomendacyjne często wykorzystują podejście typu *collaborative filtering*,
które uczy się wyłącznie na podstawie zachowań użytkowników (np. kliknięć czy zakupów)
-- bez użycia informacji o produktach.
Produkty, które pojawiają się w podobnym kontekście, otrzymują podobne reprezentacje numeryczne (osadzenia),
odzwierciedlające ich powiązania.

## Zadanie

Twoim zadaniem jest zaimplementowanie i wytrenowanie modelu,
który na podstawie osadzeń produktów pochodzących z systemu rekomendacyjnego, przewidzi **zestaw kategorii** przypisanych do każdego produktu. Zestaw ten powinien w miarę możliwości zawierać wszystkie oryginalne etykiety (niekoniecznie w oryginalnej kolejności), bez dodatkowych elementów.

## Dane

Dane składają się z osadzeń (256-wymiarowych wektorów reprezentujących produkty w przestrzeni ukrytej) i kolekcji etykiet produktów (lista list etykiet). Zostały podzielone na:

- zbiór treningowy (12145 produktów),

- zbiór walidacyjny (2603 produktów),

- tajny zbiór testowy (2603 produktów).

Kolejne wiersze w macierzy osadzeń odpowiadają tym samym produktom co kolejne elementy w liście zestawów etykiet.

## Kryterium oceny

Twoje rozwiązanie będzie oceniane przy użyciu **Intersection over Union (IoU)**, liczonego dla każdego produktu:

$$IoU = \frac{|\text{predykcja} \cap \text{prawda}|}{|\text{predykcja} \cup \text{prawda}|}$$

Metryka będzie liczona bez uwzględnienia struktury i kolejności etykiet.

Końcowy wynik to średnia wartość IoU po wszystkich produktach, przeliczona na punkty:

- jeśli wynik będzie **niższy niż 36\%**, otrzymasz **0 punktów**,

- jeśli wynik będzie **wyższy niż 46\%**, otrzymasz **maksymalną liczbę punktów**, czyli **100**.

Punktacja dla wartości pomiędzy tymi progami będzie naliczana proporcjonalnie.

## Ograniczenia

- Do uczenia modelu możesz używać jedynie zbioru treningowego.
- Twoje rozwiązanie będzie testowane na Platformie Konkursowej bez dostępu do internetu i **bez GPU**.
- Trening i ewaluacja Twojego finalnego rozwiązania na Platformie Konkursowej nie może trwać dłużej niż 5 minut.
- Dozwolone biblioteki: `numpy`, `pandas`, `sklearn`.

## Pliki zgłoszeniowe

Rozwiązaniem zadania jest ten notebook uzupełniony o Twoje rozwiązanie w postaci definicji modelu, zaimplementowanego jako specjalna klasa `YourSolution` (patrz sekcja: ___Twoje Rozwiązanie___).

## Wskazówki

- Zwróć uwagę na strukturę etykiet. Każdy produkt może należeć do **więcej niż jednej kategorii**.
- Osadzenia produktów są optymalizowane w ten sposób, że im wyższy iloczyn skalarny między osadzeniem użytkownika i osadzeniem produktu, tym wyżej produkt będzie umieszczony w spersonalizowanym rankingu.

## Ewaluacja

Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz,
będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie
wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej.
Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo,
otrzymasz za zadanie 0 punktów.

# Kod Startowy

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
import json
import numpy as np
import pandas as pd
import sklearn
import math

In [ ]:
FINAL_EVALUATION_MODE = False

## Ładowanie Danych

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
with open('data/train_categories.json', 'r') as f:
    train_categories = json.load(f)

train_embeddings = np.load('data/train_embeddings.npy', allow_pickle=True)

with open('data/val_categories.json', 'r') as f:
    val_categories = json.load(f)

val_embeddings = np.load('data/val_embeddings.npy', allow_pickle=True)

## Kod z Kryterium Oceniającym

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
def intersection_over_union(x, y):
    set_x = set(x)
    set_y = set(y)
    return len(set_x.intersection(set_y)) / len(set_x.union(set_y))

In [ ]:
def loss(true_sets, predicted_sets):
    return np.mean([intersection_over_union(x, y) for x, y in zip(true_sets, predicted_sets)])

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
def round_half_up(number: float) -> int:
    return int(math.floor(number + 0.5))

def compute_score(loss_val: float):
    upper_limit = 0.46
    lower_limit = 0.36
    if loss_val > upper_limit:
        return 100
    elif loss_val < lower_limit:
        return 0
    else:
        return round_half_up((loss_val - lower_limit) / (upper_limit - lower_limit) * 100)

# Twoje Rozwiązanie

Jako rozwiązanie należy zaimplementować klasę `YourSolution`, spełniającą pewne formalne wymogi opisane poniżej.
Definicja klasy `YourSolution` koniecznie __musi__ implementować następujące metody:

1. Metodę `fit` przyjmującą jako argument tablicę `np.ndarray`, której wiersze to 256-wymiarowe osadzenia produktów oraz listę, której elementami są listy etykiet dla kolejnych produktów.

2. Metodę `predict` przyjmującą jako argument tablicę `np.ndarray`, której wiersze to 256-wymiarowe osadzenia produktów. Ma ona zwracać listę, której elementami są listy etykiet dla kolejnych produktów.

W tej sekcji zostało zawarte przykładowe rozwiązanie. Jest to bardzo prosty baseline oparty na najczęstszej etykiecie. Dzięki temu notebook wykonuje się poprawnie nawet bez edycji tej komórki.

Wolno Ci dowolnie modyfikować definicję tej klasy, tak długo, jak spełnione będą powyższe warunki. Możesz dodać potrzebne atrybuty i metody oraz modyfikować istniejące. Twoje rozwiązanie będzie ewaluowane __wyłącznie__ na podstawie klasy `YourSolution`.

In [ ]:
######################### TUTAJ JEST MIEJSCE NA TWOJE FINALNE ROZWIĄZANIE ##########################

class YourSolution:
    def __init__(self):
        """
        init powienien korzystać z hardcodowanych lub domyślnych wartości hiperparametrów
        """
        pass

    def fit(self, X, y):
        # Tutaj możesz zawrzeć przygotowanie swoich danych
        # Wytrenuj model
        pass

    def predict(self, X):
        return [['Sports & Outdoors'] for _ in X]


# Ewaluacja

Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych treningowych. Przed wysłaniem upewnij się, że cały notebook wykonuje się od początku do końca bez błędów i bez konieczności ingerencji użytkownika po wybraniu opcji "Run All".


In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
if not FINAL_EVALUATION_MODE:
    # Wytrenowanie modelu
    yourSolutionInstance = YourSolution()
    yourSolutionInstance.fit(train_embeddings, train_categories)

    # Przygotowanie predykcji
    predictions = yourSolutionInstance.predict(val_embeddings)

    # Ocena rozwiązania
    loss_val = loss(val_categories, predictions)
    print(compute_score(loss_val))